In [1]:
import pandas as pd
import numpy as np

# **Phase 1: Data Cleaning & Preprocessing**

**Step 1: Load Dataset & Verify Structure**

In [2]:
# Load dataset
df = pd.read_csv('dataset.csv')   

# Display basic info
print("✅ Dataset Loaded Successfully!")
print("📌 Shape of Data:", df.shape)
print("\n📝 First 5 Rows:")
print(df.head())

# Check for missing values
print("\n⚠️ Missing Values Per Column:")
print(df.isnull().sum())

# Check data types
print("\n🛠 Column Data Types:")
print(df.dtypes)

C:\Users\User\AppData\Local\Temp\ipykernel_21152\1095160252.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('dataset.csv')


✅ Dataset Loaded Successfully!
📌 Shape of Data: (4526654, 24)

📝 First 5 Rows:
   Index        pH      Iron   Nitrate    Chloride Lead      Zinc  \
0      0  8.332988  0.000083  8.605777  122.799772  0.0  3.434827   
1      1  6.917863  0.000081  3.734167  227.029851  0.0  1.245317   
2      2  5.443762  0.020106  3.816994  230.995630  0.0  0.528280   
3      3  7.955339  0.143988  8.224944  178.129940  0.0  4.027879   
4      4  8.091909  0.002167  9.925788  186.540872  0.0  3.807511   

            Color  Turbidity  Fluoride  ...  Chlorine     Manganese  \
0       Colorless   0.022683  0.607283  ...  3.708178  2.269945e-15   
1    Faint Yellow   0.019007  0.622874  ...  3.292038  8.024076e-07   
2    Light Yellow   0.319956  0.423423  ...  3.560224  7.007989e-02   
3  Near Colorless   0.166319  0.208454  ...  3.516907  2.468295e-02   
4    Light Yellow   0.004867  0.222912  ...  3.177849  3.296139e-03   

   Total Dissolved Solids  Source  Water Temperature  Air Temperature  \
0     

In [3]:
df.shape

(4526654, 24)

In [4]:
import pandas as pd


# Count occurrences of 0 and 1 in the "Target" column
count_zeros = (df['Target'] == 0).sum()
count_ones = (df['Target'] == 1).sum()

# Print results
print(f"Number of occurrences of 0 in the Target column: {count_zeros}")
print(f"Number of occurrences of 1 in the Target column: {count_ones}")



Number of occurrences of 0 in the Target column: 3202702
Number of occurrences of 1 in the Target column: 1323951


In [5]:
df=df.drop("Index",axis=1)

**Step 2.1: Fixing Data Types**

In [6]:
# Convert 'Lead' column to numeric (force errors='coerce' to convert non-numeric to NaN)
df["Lead"] = pd.to_numeric(df["Lead"], errors="coerce")

# Convert 'Month' to categorical 
df["Month"] = df["Month"].astype("category")

# Convert categorical text columns to category type
df["Source"] = df["Source"].astype("category")
df["Color"] = df["Color"].astype("category")

# Verify data types after conversion
print("✅ Updated Data Types:\n", df.dtypes)


✅ Updated Data Types:
 pH                         float64
Iron                       float64
Nitrate                    float64
Chloride                   float64
Lead                       float64
Zinc                       float64
Color                     category
Turbidity                  float64
Fluoride                   float64
Copper                     float64
Odor                       float64
Sulfate                    float64
Conductivity               float64
Chlorine                   float64
Manganese                  float64
Total Dissolved Solids     float64
Source                    category
Water Temperature          float64
Air Temperature            float64
Month                     category
Day                        float64
Time of Day                float64
Target                     float64
dtype: object


**Step 2.2: Handling Missing Values**

In [7]:
# Check missing values
missing_values = df.isnull().sum()

print("⚠️ Missing Values Per Column:\n", missing_values)
print("\n📌 Total Missing Values:", missing_values.sum())

# Display basic stats before handling missing values
print("\n📝 Dataset Shape Before Handling Missing Values:", df.shape)

⚠️ Missing Values Per Column:
 pH                         87993
Iron                       30268
Nitrate                    80327
Chloride                  133284
Lead                       20380
Zinc                      118656
Color                       4347
Turbidity                  37925
Fluoride                  143819
Copper                    151897
Odor                      135592
Sulfate                   149935
Conductivity              124734
Chlorine                   43995
Manganese                  83288
Total Dissolved Solids      1285
Source                     67087
Water Temperature         127378
Air Temperature            22753
Month                      72541
Day                        75838
Time of Day                86730
Target                         1
dtype: int64

📌 Total Missing Values: 1800053

📝 Dataset Shape Before Handling Missing Values: (4526654, 23)


In [8]:
# Drop rows where Target is missing (since it's only 1 row)
df = df.dropna(subset=['Target'])

# Fill missing values for numerical columns with median
num_cols = df.select_dtypes(include=['float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill missing values for categorical columns with mode (most frequent value)
cat_cols = df.select_dtypes(include=['category']).columns
df[cat_cols] = df[cat_cols].apply(lambda x: x.fillna(x.mode()[0]))

# Verify that missing values are handled
print("✅ Missing Values After Handling:")
print(df.isnull().sum())

# Display dataset shape after handling missing values
print("\n📌 Dataset Shape After Handling Missing Values:", df.shape)


✅ Missing Values After Handling:
pH                        0
Iron                      0
Nitrate                   0
Chloride                  0
Lead                      0
Zinc                      0
Color                     0
Turbidity                 0
Fluoride                  0
Copper                    0
Odor                      0
Sulfate                   0
Conductivity              0
Chlorine                  0
Manganese                 0
Total Dissolved Solids    0
Source                    0
Water Temperature         0
Air Temperature           0
Month                     0
Day                       0
Time of Day               0
Target                    0
dtype: int64

📌 Dataset Shape After Handling Missing Values: (4526653, 23)


In [9]:
from scipy.stats import zscore

# Z-score method: clip values beyond z = ±3
def apply_zscore(df, cols, threshold=3):
    for col in cols:
        z = zscore(df[col].dropna())
        mask = abs(z) > threshold
        upper = df[col][~mask].max()
        lower = df[col][~mask].min()
        df[col] = df[col].clip(lower=lower, upper=upper)
    return df

df = apply_zscore(df, ['pH', 'Chlorine', 'Air Temperature'])


In [10]:
def apply_iqr(df, cols):
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower=lower, upper=upper)
    return df

df = apply_iqr(df, ['Iron', 'Nitrate', 'Fluoride', 'Copper', 'Turbidity'])


In [11]:
# Define domain caps manually (based on WHO/EPA standards)
domain_caps = {
    'pH': (4.5, 10.5),
    'Iron': (0, 0.5),
    'Fluoride': (0, 1.5),
    'Sulfate': (0, 500),
    'Total Dissolved Solids': (0, 500),
    'Manganese': (0, 0.4)
}

def apply_domain_cap(df, cap_dict):
    for col, (lower, upper) in cap_dict.items():
        df[col] = df[col].clip(lower=lower, upper=upper)
    return df

df = apply_domain_cap(df, domain_caps)


In [12]:
import numpy as np

def apply_log_iqr(df, cols):
    for col in cols:
        df[col] = df[col].apply(lambda x: np.log1p(x) if x > 0 else 0)
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower=lower, upper=upper)
    return df

df = apply_log_iqr(df, ['Zinc', 'Manganese'])  # Avoid reapplying to Iron if already handled


In [13]:
def apply_percentile_clip(df, cols, lower_p=0.01, upper_p=0.99):
    for col in cols:
        lower = df[col].quantile(lower_p)
        upper = df[col].quantile(upper_p)
        df[col] = df[col].clip(lower=lower, upper=upper)
    return df

df = apply_percentile_clip(df, ['Total Dissolved Solids', 'Conductivity', 'Chloride'])


In [14]:
df.shape

(4526653, 23)

In [15]:
# Save the cleaned DataFrame to CSV
df.to_csv("cleaned_dataset.csv", index=False)

**Step 2.3: Handling Invalid Values**

In [16]:
# Find negative values
invalid_values = df[num_cols][df[num_cols] < 0]

# Print results
print("⚠️ Invalid (Negative) Values Found:")
print(invalid_values.describe())


print("\n✅ Invalid Value Check Completed!")


⚠️ Invalid (Negative) Values Found:
        pH  Iron  Nitrate  Chloride  Lead  Zinc  Turbidity  Fluoride  Copper  \
count  0.0   0.0      0.0       0.0   0.0   0.0        0.0       0.0     0.0   
mean   NaN   NaN      NaN       NaN   NaN   NaN        NaN       NaN     NaN   
std    NaN   NaN      NaN       NaN   NaN   NaN        NaN       NaN     NaN   
min    NaN   NaN      NaN       NaN   NaN   NaN        NaN       NaN     NaN   
25%    NaN   NaN      NaN       NaN   NaN   NaN        NaN       NaN     NaN   
50%    NaN   NaN      NaN       NaN   NaN   NaN        NaN       NaN     NaN   
75%    NaN   NaN      NaN       NaN   NaN   NaN        NaN       NaN     NaN   
max    NaN   NaN      NaN       NaN   NaN   NaN        NaN       NaN     NaN   

       Odor  Sulfate  Conductivity  Chlorine  Manganese  \
count   0.0      0.0           0.0       0.0        0.0   
mean    NaN      NaN           NaN       NaN        NaN   
std     NaN      NaN           NaN       NaN        NaN   
min    

**One-Hot Encoding**

In [17]:
# Categorical Columns
categorical_cols = ["Color", "Source", "Month"]

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)  # Drop first category to avoid dummy trap

# Check Updated Data
print("✅ One-Hot Encoding Completed!")
print("📌 Updated Dataset Shape:", df_encoded.shape)
print("📝 First 5 Rows After Encoding:\n", df_encoded.head())


✅ One-Hot Encoding Completed!
📌 Updated Dataset Shape: (4526653, 42)
📝 First 5 Rows After Encoding:
          pH      Iron   Nitrate    Chloride           Lead      Zinc  \
0  8.332988  0.000083  8.605777  122.799772   3.713298e-52  1.489489   
1  6.917863  0.000081  3.734167  227.029851   7.849262e-94  0.808847   
2  5.443762  0.020106  3.816994  230.995630   5.286616e-76  0.424143   
3  7.955339  0.132427  8.224944  178.129940  3.997118e-176  1.614998   
4  8.091909  0.002167  9.925788  186.540872  4.171069e-132  1.570180   

   Turbidity  Fluoride    Copper      Odor  ...  Month_December  \
0   0.022683  0.607283  0.144599  1.626212  ...           False   
1   0.019007  0.622874  0.437835  1.686049  ...           False   
2   0.319956  0.423423  0.431588  3.414619  ...           False   
3   0.166319  0.208454  0.239451  1.769302  ...           False   
4   0.004867  0.222912  0.616574  0.795310  ...           False   

   Month_February  Month_January  Month_July  Month_June  Month

In [18]:
df_encoded.to_csv("cleaned_dataset_encoded.csv", index=False)

## Feature Selection

**Information Gain Calculation**

In [19]:
import pandas as pd
#  Use the Correct DataFrame (After Encoding)
df_encoded = pd.read_csv("cleaned_dataset_encoded.csv")
X = df_encoded.drop(columns=["Target"])  # All features except Target
y = df_encoded["Target"]  # Target variable

In [20]:
from sklearn.feature_selection import mutual_info_classif

# Compute Information Gain (IG)
ig_scores = mutual_info_classif(X, y, random_state=42, n_jobs=-1)  # Parallel Processing

# Create a DataFrame to Store IG Values
ig_df = pd.DataFrame({"Feature": X.columns, "Information Gain": ig_scores})

#  Sort Features by IG Score (Descending Order)
ig_df = ig_df.sort_values(by="Information Gain", ascending=False).reset_index(drop=True)

#  Display Results
print("✅ IG Calculation Completed!")
print(ig_df)


✅ IG Calculation Completed!
                   Feature  Information Gain
0                Manganese          0.096859
1                       pH          0.090135
2     Color_Near Colorless          0.080714
3                Turbidity          0.066122
4                     Iron          0.058184
5                   Copper          0.056246
6                 Chloride          0.054728
7                 Fluoride          0.052573
8                     Odor          0.051303
9       Color_Faint Yellow          0.046840
10                 Nitrate          0.044018
11            Color_Yellow          0.043239
12                Chlorine          0.041567
13  Total Dissolved Solids          0.037503
14      Color_Light Yellow          0.033763
15                 Sulfate          0.032840
16             Source_Well          0.030409
17             Time of Day          0.026053
18        Source_Reservoir          0.025079
19           Source_Spring          0.024893
20           Source_Ground 

**Fixed Threshold Method**

In [21]:
import numpy as np
# Use the 50th percentile (median) as the fixed threshold
fixed_threshold = np.percentile(ig_df["Information Gain"], 50)  

# Select features with IG >= fixed threshold
selected_features_fixed = ig_df.loc[ig_df["Information Gain"] >= fixed_threshold, "Feature"].values

# Display Results
print("\n✅ Features Selected by Fixed Threshold Method (Median-Based):")
print(selected_features_fixed)
print(f"📌 Number of Features Selected: {len(selected_features_fixed)}")



✅ Features Selected by Fixed Threshold Method (Median-Based):
['Manganese' 'pH' 'Color_Near Colorless' 'Turbidity' 'Iron' 'Copper'
 'Chloride' 'Fluoride' 'Odor' 'Color_Faint Yellow' 'Nitrate'
 'Color_Yellow' 'Chlorine' 'Total Dissolved Solids' 'Color_Light Yellow'
 'Sulfate' 'Source_Well' 'Time of Day' 'Source_Reservoir' 'Source_Spring'
 'Source_Ground']
📌 Number of Features Selected: 21


**Standard Deviation-Based Threshold Approach**

In [22]:
# Calculate the threshold as the standard deviation of IG values
std_threshold = ig_df["Information Gain"].std()

# Select features with IG >= standard deviation threshold
selected_features_std = ig_df.loc[ig_df["Information Gain"] >= std_threshold, "Feature"].values

# Display Results
print("\n✅ Features Selected by Standard Deviation Threshold Method:")
print(selected_features_std)
print(f"📌 Number of Features Selected: {len(selected_features_std)}")



✅ Features Selected by Standard Deviation Threshold Method:
['Manganese' 'pH' 'Color_Near Colorless' 'Turbidity' 'Iron' 'Copper'
 'Chloride' 'Fluoride' 'Odor' 'Color_Faint Yellow' 'Nitrate'
 'Color_Yellow' 'Chlorine' 'Total Dissolved Solids' 'Color_Light Yellow'
 'Sulfate' 'Source_Well' 'Time of Day' 'Source_Reservoir' 'Source_Spring'
 'Source_Ground' 'Source_Stream' 'Source_River' 'Source_Lake']
📌 Number of Features Selected: 24


**CBFS Method**

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

# Train Random Forest on the dataset
rf = RandomForestClassifier(n_estimators=100, max_features="sqrt", random_state=42, n_jobs=-1)
rf.fit(X, y)

# Apply CBFS: Select features with importance >= mean importance
cbfs = SelectFromModel(rf, threshold="mean", prefit=False)
cbfs.fit(X, y)  # Fit before selecting features
selected_features_cbfs = X.columns[cbfs.get_support()]

# Display Results
print("\n✅ Features Selected by CBFS Method:")
print(selected_features_cbfs)
print(f"📌 Number of Features Selected: {len(selected_features_cbfs)}")



✅ Features Selected by CBFS Method:
Index(['pH', 'Nitrate', 'Chloride', 'Zinc', 'Turbidity', 'Copper', 'Odor',
       'Sulfate', 'Conductivity', 'Chlorine', 'Manganese',
       'Total Dissolved Solids', 'Water Temperature', 'Air Temperature',
       'Color_Yellow'],
      dtype='object')
📌 Number of Features Selected: 15


**Data Cleaning for TABMAP**

In [24]:
import pandas as pd

# Step 1: Load dataset
df = pd.read_csv("cleaned_dataset_encoded.csv")

# Step 2: Convert True/False to 1/0
df = df.replace({True: 1, False: 0})

# Step 3: Select only desired features + Target
selected_columns = ['Manganese', 'pH' ,'Color_Near Colorless', 'Turbidity', 'Iron', 'Copper',
 'Chloride' ,'Fluoride', 'Odor', 'Color_Faint Yellow', 'Nitrate',
 'Color_Yellow', 'Chlorine', 'Total Dissolved Solids', 'Color_Light Yellow',
 'Sulfate', 'Source_Well', 'Time of Day', 'Source_Reservoir', 'Source_Spring',
 'Source_Ground', 'Target']

df = df[selected_columns]  # keep only these columns

# Step 4: Move 'Target' to the end (in case it isn’t already)
target_col = df.pop('Target')
df['Target'] = target_col

# Step 5: Add ID column at the beginning
df.insert(0, 'id', range(1, len(df) + 1))

# Step 6: Save to CSV
output_path = "Dataset_Tabmap_Fixed.csv"
df.to_csv(output_path, index=False)
print(f"\n✅ Filtered and cleaned dataset saved to: {output_path}")


C:\Users\User\AppData\Local\Temp\ipykernel_21152\2700702602.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({True: 1, False: 0})



✅ Filtered and cleaned dataset saved to: Dataset_Tabmap_Fixed.csv


In [ ]:
import pandas as pd

# Step 1: Load dataset
df = pd.read_csv("cleaned_dataset_encoded.csv")

# Step 2: Convert True/False to 1/0
df = df.replace({True: 1, False: 0})

# Step 3: Select only desired features + Target
selected_columns = ['Manganese', 'pH' ,'Color_Near Colorless', 'Turbidity' ,'Iron', 'Copper',
 'Chloride', 'Fluoride', 'Odor' ,'Color_Faint Yellow', 'Nitrate',
 'Color_Yellow', 'Chlorine', 'Total Dissolved Solids', 'Color_Light Yellow',
 'Sulfate', 'Source_Well', 'Time of Day', 'Source_Reservoir', 'Source_Spring',
 'Source_Ground' ,'Source_Stream' ,'Source_River' ,'Source_Lake','Target']

df = df[selected_columns]  # keep only these columns

# Step 4: Move 'Target' to the end (in case it isn’t already)
target_col = df.pop('Target')
df['Target'] = target_col

# Step 5: Add ID column at the beginning
df.insert(0, 'id', range(1, len(df) + 1))

# Step 6: Save to CSV
output_path = "Dataset_Tabmap_STD.csv"
df.to_csv(output_path, index=False)
print(f"\n✅ Filtered and cleaned dataset saved to: {output_path}")


C:\Users\User\AppData\Local\Temp\ipykernel_21152\1244871614.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({True: 1, False: 0})



✅ Filtered and cleaned dataset saved to: Dataset_Tabmap_STD.csv


In [26]:
import pandas as pd

# Step 1: Load dataset
df = pd.read_csv("cleaned_dataset_encoded.csv")

# Step 2: Convert True/False to 1/0
df = df.replace({True: 1, False: 0})

# Step 3: Select only desired features + Target
selected_columns = ['pH', 'Nitrate', 'Chloride', 'Zinc', 'Turbidity', 'Copper', 'Odor',
       'Sulfate', 'Conductivity', 'Chlorine', 'Manganese',
       'Total Dissolved Solids', 'Water Temperature', 'Air Temperature',
       'Color_Yellow','Target']

df = df[selected_columns]  # keep only these columns

# Step 4: Move 'Target' to the end (in case it isn’t already)
target_col = df.pop('Target')
df['Target'] = target_col

# Step 5: Add ID column at the beginning
df.insert(0, 'id', range(1, len(df) + 1))

# Step 6: Save to CSV
output_path = "Dataset_Tabmap_CBFS.csv"
df.to_csv(output_path, index=False)
print(f"\n✅ Filtered and cleaned dataset saved to: {output_path}")


C:\Users\User\AppData\Local\Temp\ipykernel_21152\102448888.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({True: 1, False: 0})



✅ Filtered and cleaned dataset saved to: Dataset_Tabmap_CBFS.csv


In [2]:
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd

# Load dataset
data = pd.read_csv('cleaned_dataset_encoded.csv')

# Separate features and target
X = data.drop(columns=['Target'])  # Features
y = data['Target']  # Target

# Apply random undersampling with 'auto' for 1:1 class balance
rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)  # Downsample to the size of the minority class
X_resampled, y_resampled = rus.fit_resample(X, y)

# Combine into a new balanced DataFrame
balanced_data = pd.DataFrame(X_resampled, columns=X.columns)
balanced_data['Target'] = y_resampled

# Save the new balanced dataset
balanced_data.to_csv('balanced_dataset_encoded.csv', index=False)

# Verify the class distribution
print(balanced_data['Target'].value_counts())  


Target
0.0    1323951
1.0    1323951
Name: count, dtype: int64


In [3]:
import pandas as pd

# Step 1: Load dataset
df = pd.read_csv("balanced_dataset_encoded.csv")

# Step 2: Convert True/False to 1/0
df = df.replace({True: 1, False: 0})

# Step 4: Move 'Target' to the end (in case it isn’t already)
target_col = df.pop('Target')
df['Target'] = target_col

# Step 5: Add ID column at the beginning
df.insert(0, 'id', range(1, len(df) + 1))

# Step 6: Save to CSV
output_path = "Dataset_Tabmap_Before.csv"
df.to_csv(output_path, index=False)
print(f"\n✅ Filtered and cleaned dataset saved to: {output_path}")


C:\Users\User\AppData\Local\Temp\ipykernel_5848\116769388.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({True: 1, False: 0})



✅ Filtered and cleaned dataset saved to: Dataset_Tabmap_Before.csv
